In [11]:

import os
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 220)

DB_DIR = Path(os.environ.get('SUJET2_DB_DIR', 'db'))
OUT_DIR = Path(os.environ.get('SUJET2_OUT_DIR', 'output'))
OUT_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    'sitehist': DB_DIR / 'sitehist.csv',
    'sitepred': DB_DIR / 'sitepred.csv',
    'siteweath': DB_DIR / 'siteweath.csv',
    'zonehist': DB_DIR / 'zonehist.csv',
    'zonepred': DB_DIR / 'zonepred.csv'
}

def read_semicolon_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep=';', quotechar='"', encoding='utf-8', engine='python')

raw = {}
for k, p in paths.items():
    if p.exists():
        raw[k] = read_semicolon_csv(p)
        print(k, 'loaded', raw[k].shape)
    else:
        print(k, 'missing', p)

sitehist loaded (3492, 15)
sitepred loaded (3573, 9)
siteweath loaded (4184, 15)
zonehist loaded (21048, 16)
zonepred loaded (20237, 10)


In [12]:

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().strip('"') for c in df.columns]
    lower_map = {c.lower(): c for c in df.columns}

    def rename_if_present(target: str, *candidates: str):
        for cand in candidates:
            if cand in lower_map:
                df.rename(columns={lower_map[cand]: target}, inplace=True)
                break

    rename_if_present('siteId', 'siteid', 'site_id', 'site id')
    rename_if_present('zoneId', 'zoneid', 'zone_id', 'zone id')
    rename_if_present('dtUpdate', 'dtupdate', 'dt_update', 'dt update')
    return df


def to_day(s):
    return pd.to_datetime(s, errors='coerce').dt.floor('D')

sitehist = normalize_columns(raw.get('sitehist', pd.DataFrame()))
sitepred = normalize_columns(raw.get('sitepred', pd.DataFrame()))
siteweath = normalize_columns(raw.get('siteweath', pd.DataFrame()))
zonehist = normalize_columns(raw.get('zonehist', pd.DataFrame()))
zonepred = normalize_columns(raw.get('zonepred', pd.DataFrame()))

if 'dtUpdate' in sitehist.columns and 'date' not in sitehist.columns:
    sitehist['date'] = to_day(sitehist['dtUpdate'])
if 'date' in sitepred.columns:
    sitepred['date'] = to_day(sitepred['date'])
if 'dtUpdate' in siteweath.columns and 'date' not in siteweath.columns:
    siteweath['date'] = to_day(siteweath['dtUpdate'])
if 'dtUpdate' in zonehist.columns and 'date' not in zonehist.columns:
    zonehist['date'] = to_day(zonehist['dtUpdate'])
if 'date' in zonepred.columns:
    zonepred['date'] = to_day(zonepred['date'])

for df in (sitehist, sitepred, siteweath, zonehist, zonepred):
    if 'siteId' in df.columns:
        df['siteId'] = pd.to_numeric(df['siteId'], errors='coerce').astype('Int64')
    if 'zoneId' in df.columns:
        df['zoneId'] = pd.to_numeric(df['zoneId'], errors='coerce').astype('Int64')
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.floor('D')

print('keys ok:', 'siteId' in sitehist.columns, 'date' in sitehist.columns)

keys ok: True True


In [13]:

ZERO_OR_NEG_IS_MISSING = {
    'elecTotalKwh': True,
    'totalKwh': True,
    'waterM3': True,
    'totalWater': True,
    'totalOptimValue': True,
    'value': True
}

UNIT_FIX_RULES = {
    'elecTotalKwh': [1000, 1_000_000],
    'totalKwh': [1000, 1_000_000],
    'waterM3': [1000],
    'totalWater': [1000]
}

CUMUL_SPIKE = {
    'min_missing_run': 3,
    'spike_factor': 20.0,
    'strategy': 'spread',
    'baseline_points': 30,
    'max_spread_days': 370
}

MEASURE_COLS_SITEHIST = [c for c in ['elecTotalKwh', 'waterM3', 'totalOptimValue', 'elecCvcKwh', 'elecBveKwh', 'elecLightingKwh', 'elecForceKwh', 'elecAggregatedKwh'] if c in sitehist.columns]
MEASURE_COLS_ZONEHIST = [c for c in ['elecTotalKwh', 'waterM3', 'totalOptimValue', 'elecCvcKwh', 'elecBveKwh', 'elecLightingKwh', 'elecForceKwh', 'elecAggregatedKwh', 'indoorTempDegC'] if c in zonehist.columns]
MEASURE_COLS_SITEPRED = [c for c in ['totalKwh', 'totalWater', 'value', 'tempAmb'] if c in sitepred.columns]
MEASURE_COLS_ZONEPRED = [c for c in ['totalKwh', 'totalWater', 'value', 'tempAmb'] if c in zonepred.columns]

print('measure cols sitehist', MEASURE_COLS_SITEHIST)

measure cols sitehist ['elecTotalKwh', 'waterM3', 'totalOptimValue', 'elecCvcKwh', 'elecBveKwh', 'elecLightingKwh', 'elecForceKwh', 'elecAggregatedKwh']


In [14]:

def _num(s):
    return pd.to_numeric(s, errors='coerce')


def apply_missing_sentinels(df: pd.DataFrame, cols, rule_map: dict) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        x = _num(df[c])
        if rule_map.get(c, False):
            x = x.mask(x <= 0, np.nan)
        else:
            x = x.mask(x < 0, np.nan)
        df[c] = x
    return df


def fix_unit_outliers(df: pd.DataFrame, group_cols, value_col: str, divisors, factor=200.0, min_med=1e-9):
    df = df.copy()
    log = []
    if value_col not in df.columns:
        return df, pd.DataFrame(log)

    df[value_col] = _num(df[value_col])
    med = (df.loc[df[value_col].notna() & (df[value_col] > 0)]
             .groupby(group_cols)[value_col].median())
    med_map = med.to_dict()

    vals = df[value_col].to_numpy()
    new_vals = vals.copy()

    for i in range(len(df)):
        v = vals[i]
        if not np.isfinite(v) or v <= 0:
            continue
        k = tuple(df.iloc[i][group_cols].values.tolist())
        m = med_map.get(k, np.nan)
        if not np.isfinite(m) or m < min_med:
            continue
        if v / m <= factor:
            continue

        best = None
        best_ratio = v / m
        for d in divisors:
            vd = v / d
            rd = vd / m
            if rd < best_ratio:
                best_ratio = rd
                best = (d, vd)
        if best is not None and best_ratio <= factor:
            d, vd = best
            new_vals[i] = vd
            log.append({'value_col': value_col, 'group': k, 'index': int(i), 'old': float(v), 'new': float(vd), 'divisor': int(d)})

    df[value_col] = new_vals
    return df, pd.DataFrame(log)


def spread_cumul_spikes_v2(df: pd.DataFrame, group_cols, date_col: str, value_col: str, cfg: dict):
    df = df.copy()
    logs = []
    if value_col not in df.columns or date_col not in df.columns:
        return df, pd.DataFrame(logs)

    df[value_col] = _num(df[value_col])

    min_run = int(cfg.get('min_missing_run', 3))
    spike_factor = float(cfg.get('spike_factor', 20.0))
    strategy = cfg.get('strategy', 'spread')
    baseline_points = int(cfg.get('baseline_points', 30))
    max_spread_days = int(cfg.get('max_spread_days', 370))

    out_parts = []
    for keys, g in df.groupby(group_cols, dropna=False):
        g = g.sort_values(date_col)
        if g[date_col].isna().all():
            out_parts.append(g)
            continue

        idx = pd.date_range(g[date_col].min(), g[date_col].max(), freq='D')
        g2 = g.set_index(date_col).reindex(idx)
        g2.index.name = date_col

        if not isinstance(keys, tuple):
            keys = (keys,)
        for c, v in zip(group_cols, keys):
            g2[c] = v

        x = g2[value_col].copy()
        valid = x.notna() & (x > 0)
        is_missing = ~valid

        overall_med = float(np.nanmedian(x[valid].to_numpy())) if valid.any() else np.nan

        last_valid_pos = np.full(len(x), -1, dtype=int)
        last = -1
        for i in range(len(x)):
            last_valid_pos[i] = last
            if valid.iloc[i]:
                last = i

        for i in range(len(x)):
            v = x.iloc[i]
            if not np.isfinite(v) or v <= 0:
                continue
            prev = last_valid_pos[i]
            if prev < 0:
                continue
            gap = i - prev - 1
            if gap < min_run:
                continue

            prev_vals = x.iloc[:prev + 1]
            prev_valid = prev_vals[prev_vals.notna() & (prev_vals > 0)]
            baseline = float(np.nanmedian(prev_valid.tail(baseline_points).to_numpy())) if len(prev_valid) else np.nan
            if not np.isfinite(baseline) or baseline <= 0:
                baseline = overall_med
            if not np.isfinite(baseline) or baseline <= 0:
                continue

            if v <= spike_factor * baseline:
                continue

            span = gap + 1
            action = strategy
            if span > max_spread_days and strategy == 'spread':
                action = 'drop'

            if action == 'drop':
                x.iloc[i] = np.nan
                logs.append({'group': keys, 'value_col': value_col, 'date': str(g2.index[i].date()), 'action': 'drop', 'old': float(v), 'gap_days': int(gap), 'span_days': int(span), 'baseline': float(baseline)})
            else:
                per_day = v / span
                start = i - gap
                for j in range(start, i + 1):
                    if j == i or is_missing.iloc[j]:
                        x.iloc[j] = per_day
                logs.append({'group': keys, 'value_col': value_col, 'date': str(g2.index[i].date()), 'action': 'spread', 'old': float(v), 'new_per_day': float(per_day), 'gap_days': int(gap), 'span_days': int(span), 'baseline': float(baseline)})

        g2[value_col] = x
        out_parts.append(g2.reset_index())

    out = pd.concat(out_parts, ignore_index=True)
    out = out[[c for c in df.columns if c in out.columns]]
    return out, pd.DataFrame(logs)


def find_long_gaps(df: pd.DataFrame, group_col: str, date_col: str, value_col: str, min_gap_days=30):
    if group_col not in df.columns or date_col not in df.columns or value_col not in df.columns:
        return pd.DataFrame([])
    d = df[[group_col, date_col, value_col]].copy()
    d[date_col] = pd.to_datetime(d[date_col], errors='coerce').dt.floor('D')
    d[value_col] = _num(d[value_col])
    out = []
    for gid, g in d.groupby(group_col, dropna=False):
        g = g.dropna(subset=[date_col]).sort_values(date_col)
        if g.empty:
            continue
        idx = pd.date_range(g[date_col].min(), g[date_col].max(), freq='D')
        s = g.set_index(date_col)[value_col].reindex(idx)
        missing = s.isna() | (s <= 0)
        run = 0
        run_start = None
        for dt, m in missing.items():
            if m:
                if run == 0:
                    run_start = dt
                run += 1
            else:
                if run >= min_gap_days:
                    out.append({group_col: gid, 'value_col': value_col, 'gap_start': str(run_start.date()), 'gap_end': str((dt - pd.Timedelta(days=1)).date()), 'gap_days': int(run)})
                run = 0
                run_start = None
        if run >= min_gap_days and run_start is not None:
            out.append({group_col: gid, 'value_col': value_col, 'gap_start': str(run_start.date()), 'gap_end': str(idx.max().date()), 'gap_days': int(run)})
    return pd.DataFrame(out)

In [15]:

CLEAN_LOGS = {}

sitehist = apply_missing_sentinels(sitehist, MEASURE_COLS_SITEHIST, ZERO_OR_NEG_IS_MISSING)
sitepred = apply_missing_sentinels(sitepred, MEASURE_COLS_SITEPRED, ZERO_OR_NEG_IS_MISSING)
zonehist = apply_missing_sentinels(zonehist, MEASURE_COLS_ZONEHIST, ZERO_OR_NEG_IS_MISSING)
zonepred = apply_missing_sentinels(zonepred, MEASURE_COLS_ZONEPRED, ZERO_OR_NEG_IS_MISSING)

for df in (sitehist, sitepred, siteweath, zonehist, zonepred):
    if 'siteId' in df.columns:
        df['siteId'] = pd.to_numeric(df['siteId'], errors='coerce').astype('Int64')
    if 'zoneId' in df.columns:
        df['zoneId'] = pd.to_numeric(df['zoneId'], errors='coerce').astype('Int64')
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce').dt.floor('D')

if set(['siteId']).issubset(sitehist.columns):
    for col, divs in UNIT_FIX_RULES.items():
        if col in sitehist.columns:
            sitehist, log = fix_unit_outliers(sitehist, ['siteId'], col, divs)
            if len(log):
                CLEAN_LOGS[f'sitehist_unit_{col}'] = log
if set(['siteId']).issubset(sitepred.columns):
    for col, divs in UNIT_FIX_RULES.items():
        if col in sitepred.columns:
            sitepred, log = fix_unit_outliers(sitepred, ['siteId'], col, divs)
            if len(log):
                CLEAN_LOGS[f'sitepred_unit_{col}'] = log
if set(['siteId', 'zoneId']).issubset(zonehist.columns):
    for col, divs in UNIT_FIX_RULES.items():
        if col in zonehist.columns:
            zonehist, log = fix_unit_outliers(zonehist, ['siteId', 'zoneId'], col, divs)
            if len(log):
                CLEAN_LOGS[f'zonehist_unit_{col}'] = log
if set(['siteId', 'zoneId']).issubset(zonepred.columns):
    for col, divs in UNIT_FIX_RULES.items():
        if col in zonepred.columns:
            zonepred, log = fix_unit_outliers(zonepred, ['siteId', 'zoneId'], col, divs)
            if len(log):
                CLEAN_LOGS[f'zonepred_unit_{col}'] = log

if set(['siteId', 'date']).issubset(sitehist.columns):
    for col in [c for c in MEASURE_COLS_SITEHIST if c in ('elecTotalKwh', 'waterM3', 'elecCvcKwh', 'elecBveKwh', 'elecLightingKwh', 'elecForceKwh', 'elecAggregatedKwh')]:
        sitehist, log = spread_cumul_spikes_v2(sitehist, ['siteId'], 'date', col, CUMUL_SPIKE)
        if len(log):
            CLEAN_LOGS[f'sitehist_cumul_{col}'] = log

if set(['siteId', 'zoneId', 'date']).issubset(zonehist.columns):
    for col in [c for c in MEASURE_COLS_ZONEHIST if c in ('elecTotalKwh', 'waterM3', 'elecCvcKwh', 'elecBveKwh', 'elecLightingKwh', 'elecForceKwh', 'elecAggregatedKwh')]:
        zonehist, log = spread_cumul_spikes_v2(zonehist, ['siteId', 'zoneId'], 'date', col, CUMUL_SPIKE)
        if len(log):
            CLEAN_LOGS[f'zonehist_cumul_{col}'] = log

print('clean logs', {k: len(v) for k, v in CLEAN_LOGS.items()})

clean logs {'zonehist_unit_elecTotalKwh': 52, 'zonehist_unit_waterM3': 4, 'zonepred_unit_totalKwh': 45, 'zonepred_unit_totalWater': 31, 'sitehist_cumul_elecTotalKwh': 10, 'sitehist_cumul_waterM3': 4, 'sitehist_cumul_elecCvcKwh': 7, 'sitehist_cumul_elecBveKwh': 2, 'sitehist_cumul_elecLightingKwh': 5, 'sitehist_cumul_elecForceKwh': 6, 'sitehist_cumul_elecAggregatedKwh': 11, 'zonehist_cumul_elecTotalKwh': 28, 'zonehist_cumul_waterM3': 8, 'zonehist_cumul_elecCvcKwh': 24, 'zonehist_cumul_elecBveKwh': 2, 'zonehist_cumul_elecLightingKwh': 7, 'zonehist_cumul_elecForceKwh': 14, 'zonehist_cumul_elecAggregatedKwh': 35}


In [16]:

GAPS = []
if set(['siteId', 'date']).issubset(sitehist.columns):
    for col in [c for c in MEASURE_COLS_SITEHIST if c in ('elecTotalKwh', 'waterM3')]:
        g = find_long_gaps(sitehist, 'siteId', 'date', col, min_gap_days=30)
        if len(g):
            GAPS.append(g)
if len(GAPS):
    gaps = pd.concat(GAPS, ignore_index=True).sort_values('gap_days', ascending=False)
else:
    gaps = pd.DataFrame([])

display(gaps.head(50))

,siteId,value_col,gap_start,gap_end,gap_days
9,160,waterM3,2024-07-10,2026-04-02,632
6,130,waterM3,2024-06-30,2025-10-26,484
11,180,waterM3,2025-01-09,2026-04-02,449
8,150,waterM3,2024-06-16,2025-01-18,217
12,190,waterM3,2025-05-02,2025-11-12,195
1,130,elecTotalKwh,2025-11-13,2026-04-02,141
7,130,waterM3,2025-11-13,2026-04-02,141
2,160,elecTotalKwh,2024-08-06,2024-11-26,113
14,230,waterM3,2025-12-16,2026-04-02,108
0,130,elecTotalKwh,2024-03-14,2024-06-03,82


In [17]:

site_join_cols = ['siteId', 'date']
zone_join_cols = ['siteId', 'zoneId', 'date']

sitehist_d = sitehist.dropna(subset=site_join_cols).drop_duplicates(subset=site_join_cols, keep='last')
sitepred_d = sitepred.dropna(subset=site_join_cols).drop_duplicates(subset=site_join_cols, keep='last')

site_base = sitepred_d.merge(sitehist_d, on=site_join_cols, how='left', suffixes=('_pred', '_hist'))

if len(siteweath) and set(site_join_cols).issubset(siteweath.columns):
    siteweath_d = siteweath.dropna(subset=site_join_cols).drop_duplicates(subset=site_join_cols, keep='last')
    w = siteweath_d.rename(columns={c: f'weath_{c}' for c in siteweath_d.columns if c not in site_join_cols})
    site_base = site_base.merge(w, on=site_join_cols, how='left')

zonehist_d = zonehist.dropna(subset=zone_join_cols).drop_duplicates(subset=zone_join_cols, keep='last')
zonepred_d = zonepred.dropna(subset=zone_join_cols).drop_duplicates(subset=zone_join_cols, keep='last')

zone_base = zonepred_d.merge(zonehist_d, on=zone_join_cols, how='left', suffixes=('_pred', '_hist'))

print('site_base', site_base.shape)
print('zone_base', zone_base.shape)

site_base (3572, 37)
zone_base (20237, 24)


In [18]:

def mae(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.mean(np.abs(y_true[m] - y_pred[m]))) if m.any() else np.nan


def rmse(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(np.mean((y_true[m] - y_pred[m]) ** 2))) if m.any() else np.nan


def mape(y_true, y_pred, eps=1e-9):
    m = np.isfinite(y_true) & np.isfinite(y_pred) & (np.abs(y_true) > eps)
    return float(100.0 * np.mean(np.abs((y_true[m] - y_pred[m]) / y_true[m]))) if m.any() else np.nan


def bias(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.mean(y_pred[m] - y_true[m])) if m.any() else np.nan


def metrics_row(df, y_true_col, y_pred_col):
    y_true_obj = df[y_true_col]
    y_pred_obj = df[y_pred_col]
    if isinstance(y_true_obj, pd.DataFrame):
        y_true_obj = y_true_obj.iloc[:, 0]
    if isinstance(y_pred_obj, pd.DataFrame):
        y_pred_obj = y_pred_obj.iloc[:, 0]
    y_true = pd.to_numeric(y_true_obj, errors='coerce').to_numpy(dtype=float)
    y_pred = pd.to_numeric(y_pred_obj, errors='coerce').to_numpy(dtype=float)
    return {
        'n': int(np.isfinite(y_true).sum()),
        'MAE': mae(y_true, y_pred),
        'RMSE': rmse(y_true, y_pred),
        'MAPE_%': mape(y_true, y_pred),
        'Bias(pred-true)': bias(y_true, y_pred)
    }


def summarize_by_group(df, y_true_col, y_pred_col, group_cols):
    rows = []
    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        d = metrics_row(g, y_true_col, y_pred_col)
        rows.append((*keys, d['n'], d['MAE'], d['RMSE'], d['MAPE_%'], d['Bias(pred-true)']))
    cols = list(group_cols) + ['n', 'MAE', 'RMSE', 'MAPE_%', 'Bias(pred-true)']
    return pd.DataFrame(rows, columns=cols)

SITE_MAP = {
    'totalKwh': ('elecTotalKwh', 'site'),
    'totalWater': ('waterM3', 'site'),
    'tempAmb': ('weath_tempAmb', 'site'),
    'value': ('totalOptimValue', 'site')
}

ZONE_MAP = {
    'totalKwh': ('elecTotalKwh', 'zone'),
    'totalWater': ('waterM3', 'zone'),
    'tempAmb': ('indoorTempDegC', 'zone'),
    'value': ('totalOptimValue', 'zone')
}

SITE_MAP = {k: v for k, v in SITE_MAP.items() if k in site_base.columns and v[0] in site_base.columns}
ZONE_MAP = {k: v for k, v in ZONE_MAP.items() if k in zone_base.columns and v[0] in zone_base.columns}

site_results = []
site_by_site = {}
for pred_col, (true_col, _) in SITE_MAP.items():
    g = site_base.dropna(subset=['siteId', 'date'])
    g = g[['siteId', 'date', pred_col, true_col]].copy()
    d = metrics_row(g, true_col, pred_col)
    site_results.append({'level': 'site', 'pred_col': pred_col, 'true_col': true_col, **d})
    by = summarize_by_group(g, true_col, pred_col, ['siteId'])
    by.insert(0, 'pred_col', pred_col)
    by.insert(1, 'true_col', true_col)
    site_by_site[pred_col] = by.sort_values('RMSE', ascending=False)

zone_results = []
zone_by_zone = {}
for pred_col, (true_col, _) in ZONE_MAP.items():
    g = zone_base.dropna(subset=['siteId', 'zoneId', 'date'])
    g = g[['siteId', 'zoneId', 'date', pred_col, true_col]].copy()
    d = metrics_row(g, true_col, pred_col)
    zone_results.append({'level': 'zone', 'pred_col': pred_col, 'true_col': true_col, **d})
    by = summarize_by_group(g, true_col, pred_col, ['siteId', 'zoneId'])
    by.insert(0, 'pred_col', pred_col)
    by.insert(1, 'true_col', true_col)
    zone_by_zone[pred_col] = by.sort_values('RMSE', ascending=False)

site_results = pd.DataFrame(site_results).sort_values('RMSE', ascending=False)
zone_results = pd.DataFrame(zone_results).sort_values('RMSE', ascending=False)
summary = pd.concat([site_results, zone_results], ignore_index=True)

display(summary)

,level,pred_col,true_col,n,MAE,RMSE,MAPE_%,Bias(pred-true)
0,site,totalKwh,elecTotalKwh,3018,1.155117e+06,6.095563e+07,4.722991e+03,-1.122301e+06
1,site,totalWater,waterM3,1218,6.575409e+04,3.405023e+05,4.609655e+07,6.575408e+04
2,site,tempAmb,weath_tempAmb,3482,8.602488e+00,1.053260e+01,1.484363e+02,-2.826616e+00
3,site,value,totalOptimValue,462,NaN,NaN,NaN,NaN
4,zone,totalWater,waterM3,2253,3.039306e+03,3.494893e+04,2.721172e+07,3.039249e+03
5,zone,totalKwh,elecTotalKwh,17303,1.498003e+02,1.490677e+03,1.495239e+05,2.424869e+01
6,zone,tempAmb,indoorTempDegC,18002,3.075513e+00,6.743173e+00,2.685293e+01,-1.820650e-01
7,zone,value,totalOptimValue,2371,NaN,NaN,NaN,NaN


In [19]:

summary.to_csv(OUT_DIR / 'metrics_summary_all_metrics_cleaned.csv', index=False)
for metric, df in site_by_site.items():
    df.to_csv(OUT_DIR / f'metrics_by_site_{metric}_cleaned.csv', index=False)
for metric, df in zone_by_zone.items():
    df.to_csv(OUT_DIR / f'metrics_by_zone_{metric}_cleaned.csv', index=False)
for k, log in CLEAN_LOGS.items():
    log.to_csv(OUT_DIR / f'cleanlog_{k}.csv', index=False)
if len(gaps):
    gaps.to_csv(OUT_DIR / 'data_gaps_by_site.csv', index=False)
print('wrote to', OUT_DIR)

wrote to output


In [20]:

def safe_mean(series):
    x = pd.to_numeric(series, errors='coerce')
    x = x[np.isfinite(x)]
    return float(x.mean()) if len(x) else np.nan

metric_stats = []
for pred_col, (true_col, _) in SITE_MAP.items():
    mu = safe_mean(site_base[true_col])
    metric_stats.append({'level': 'site', 'pred_col': pred_col, 'true_col': true_col, 'true_mean': mu})
for pred_col, (true_col, _) in ZONE_MAP.items():
    mu = safe_mean(zone_base[true_col])
    metric_stats.append({'level': 'zone', 'pred_col': pred_col, 'true_col': true_col, 'true_mean': mu})
metric_stats = pd.DataFrame(metric_stats)

summary2 = summary.merge(metric_stats, on=['level', 'pred_col', 'true_col'], how='left')
summary2['NRMSE'] = summary2['RMSE'] / summary2['true_mean']
summary2['bias_sign'] = np.where(summary2['Bias(pred-true)'] > 0, 'over', np.where(summary2['Bias(pred-true)'] < 0, 'under', 'neutral'))

display(summary2.sort_values('NRMSE', ascending=True))

if len(gaps):
    print('top gaps')
    display(gaps.head(20))

,level,pred_col,true_col,n,MAE,RMSE,MAPE_%,Bias(pred-true),true_mean,NRMSE,bias_sign
2,site,tempAmb,weath_tempAmb,3482,8.602488e+00,1.053260e+01,1.484363e+02,-2.826616e+00,1.369380e+01,0.769151,under
6,zone,tempAmb,indoorTempDegC,18002,3.075513e+00,6.743173e+00,2.685293e+01,-1.820650e-01,7.948132e+00,0.848397,under
5,zone,totalKwh,elecTotalKwh,17303,1.498003e+02,1.490677e+03,1.495239e+05,2.424869e+01,2.282454e+02,6.531027,over
0,site,totalKwh,elecTotalKwh,3018,1.155117e+06,6.095563e+07,4.722991e+03,-1.122301e+06,1.111481e+06,54.841790,under
4,zone,totalWater,waterM3,2253,3.039306e+03,3.494893e+04,2.721172e+07,3.039249e+03,1.358161e+00,25732.529198,over
1,site,totalWater,waterM3,1218,6.575409e+04,3.405023e+05,4.609655e+07,6.575408e+04,5.509338e+00,61804.566476,over
3,site,value,totalOptimValue,462,NaN,NaN,NaN,NaN,4.080395e+00,NaN,neutral
7,zone,value,totalOptimValue,2371,NaN,NaN,NaN,NaN,3.914204e+01,NaN,neutral


top gaps


,siteId,value_col,gap_start,gap_end,gap_days
9,160,waterM3,2024-07-10,2026-04-02,632
6,130,waterM3,2024-06-30,2025-10-26,484
11,180,waterM3,2025-01-09,2026-04-02,449
8,150,waterM3,2024-06-16,2025-01-18,217
12,190,waterM3,2025-05-02,2025-11-12,195
1,130,elecTotalKwh,2025-11-13,2026-04-02,141
7,130,waterM3,2025-11-13,2026-04-02,141
2,160,elecTotalKwh,2024-08-06,2024-11-26,113
14,230,waterM3,2025-12-16,2026-04-02,108
0,130,elecTotalKwh,2024-03-14,2024-06-03,82


In [21]:

def top_bottom(df, nmin, top=10):
    d = df.copy()
    d['n'] = pd.to_numeric(d['n'], errors='coerce')
    d['RMSE'] = pd.to_numeric(d['RMSE'], errors='coerce')
    d = d[(d['n'] >= nmin) & np.isfinite(d['RMSE']) & (d['RMSE'] > 0)]
    return d.sort_values('RMSE', ascending=False).head(top), d.sort_values('RMSE', ascending=True).head(top)

if 'totalKwh' in site_by_site:
    w, b = top_bottom(site_by_site['totalKwh'], 30)
    print('totalKwh sites worst')
    display(w[['siteId', 'n', 'RMSE', 'MAE', 'Bias(pred-true)']])
    print('totalKwh sites best')
    display(b[['siteId', 'n', 'RMSE', 'MAE', 'Bias(pred-true)']])

if 'totalWater' in site_by_site:
    w, b = top_bottom(site_by_site['totalWater'], 10)
    print('totalWater sites worst')
    display(w[['siteId', 'n', 'RMSE', 'MAE', 'Bias(pred-true)']])
    print('totalWater sites best')
    display(b[['siteId', 'n', 'RMSE', 'MAE', 'Bias(pred-true)']])

if 'tempAmb' in site_by_site:
    w, b = top_bottom(site_by_site['tempAmb'], 30)
    print('tempAmb sites worst')
    display(w[['siteId', 'n', 'RMSE', 'MAE', 'Bias(pred-true)']])
    print('tempAmb sites best')
    display(b[['siteId', 'n', 'RMSE', 'MAE', 'Bias(pred-true)']])

totalKwh sites worst


,siteId,n,RMSE,MAE,Bias(pred-true)
1,150,618,749645.938378,118935.668141,5472.062680
5,190,268,131587.473421,34967.062603,34731.043021
4,180,332,83560.670759,6614.837134,-3030.095242
7,210,192,8948.567984,702.698640,-619.765161
2,160,304,4466.945442,1131.800202,303.037472
8,220,77,2021.854131,713.333106,79.801201
0,130,413,689.151172,151.580779,74.671354
6,200,227,347.355274,166.630034,30.953906
3,170,412,181.332837,91.486066,-29.102406
10,260,48,166.139678,63.758787,-8.581878


totalKwh sites best


,siteId,n,RMSE,MAE,Bias(pred-true)
9,230,94,75.805302,37.467220,5.553656
10,260,48,166.139678,63.758787,-8.581878
3,170,412,181.332837,91.486066,-29.102406
6,200,227,347.355274,166.630034,30.953906
0,130,413,689.151172,151.580779,74.671354
8,220,77,2021.854131,713.333106,79.801201
2,160,304,4466.945442,1131.800202,303.037472
7,210,192,8948.567984,702.698640,-619.765161
4,180,332,83560.670759,6614.837134,-3030.095242
5,190,268,131587.473421,34967.062603,34731.043021


totalWater sites worst


...

,siteId,n,RMSE,MAE,Bias(pred-true)
7,210,151,5.392992e+01,28.151541,28.151541
0,130,49,1.398273e+02,74.495305,74.362014
3,170,339,4.641293e+02,242.949893,242.949893
6,200,161,1.391223e+03,1071.402625,1071.402625
1,150,411,4.961881e+03,1547.463824,1547.463824
11,270,25,3.836772e+04,20456.372963,20456.372963
8,220,69,1.134398e+06,719276.744029,719276.744029


tempAmb sites worst


,siteId,n,RMSE,MAE,Bias(pred-true)
5,190,297,17.457643,16.051178,-16.051178
6,200,237,14.317785,12.880970,-12.880970
11,270,49,13.485264,13.166122,-13.166122
7,210,200,12.874364,11.784550,-11.743050
10,260,80,11.662575,11.120625,-11.120625
8,220,87,11.604977,11.095057,-11.095057
2,160,376,10.670738,8.048454,-4.384791
9,230,106,10.660549,9.714245,-9.630094
3,170,435,9.753186,8.345775,-0.481005
0,130,531,8.443303,7.145370,6.216808


tempAmb sites best


,siteId,n,RMSE,MAE,Bias(pred-true)
1,150,637,5.460112,4.258478,2.973652
4,180,367,7.862663,6.534376,0.643719
12,280,64,8.249527,7.774749,7.211312
0,130,531,8.443303,7.145370,6.216808
3,170,435,9.753186,8.345775,-0.481005
9,230,106,10.660549,9.714245,-9.630094
2,160,376,10.670738,8.048454,-4.384791
8,220,87,11.604977,11.095057,-11.095057
10,260,80,11.662575,11.120625,-11.120625
7,210,200,12.874364,11.784550,-11.743050


In [22]:

log_counts = pd.DataFrame([{'action': k, 'count': len(v)} for k, v in CLEAN_LOGS.items()]).sort_values('count', ascending=False)
print('log counts')
display(log_counts)

log counts


,action,count
0,zonehist_unit_elecTotalKwh,52
2,zonepred_unit_totalKwh,45
17,zonehist_cumul_elecAggregatedKwh,35
3,zonepred_unit_totalWater,31
11,zonehist_cumul_elecTotalKwh,28
13,zonehist_cumul_elecCvcKwh,24
16,zonehist_cumul_elecForceKwh,14
10,sitehist_cumul_elecAggregatedKwh,11
4,sitehist_cumul_elecTotalKwh,10
12,zonehist_cumul_waterM3,8


In [23]:

def verdict(row):
    n = row['n']
    nrmse = row['NRMSE']
    if not np.isfinite(nrmse) or not np.isfinite(n):
        return 'insufficient'
    if n < 200:
        return 'limited'
    if nrmse <= 0.2:
        return 'strong'
    if nrmse <= 0.5:
        return 'ok'
    return 'weak'

summary2['verdict'] = summary2.apply(verdict, axis=1)

print('auto conclusions')
for _, r in summary2.sort_values(['level', 'NRMSE']).iterrows():
    n = int(r['n']) if np.isfinite(r['n']) else r['n']
    print(f"{r['level']} {r['pred_col']} vs {r['true_col']}: verdict={r['verdict']} bias={r['bias_sign']} n={n} NRMSE={r['NRMSE']}")

if len(gaps):
    print('gap conclusions (top 10)')
    for _, g in gaps.head(10).iterrows():
        print(f"siteId={g['siteId']} value_col={g['value_col']} gap_days={g['gap_days']} {g['gap_start']} -> {g['gap_end']}")

auto conclusions
site tempAmb vs weath_tempAmb: verdict=weak bias=under n=3482 NRMSE=0.7691505868503536
site totalKwh vs elecTotalKwh: verdict=weak bias=under n=3018 NRMSE=54.84178987649647
site totalWater vs waterM3: verdict=weak bias=over n=1218 NRMSE=61804.56647558112
site value vs totalOptimValue: verdict=insufficient bias=neutral n=462 NRMSE=nan
zone tempAmb vs indoorTempDegC: verdict=weak bias=under n=18002 NRMSE=0.8483972390050907
zone totalKwh vs elecTotalKwh: verdict=weak bias=over n=17303 NRMSE=6.5310269428285
zone totalWater vs waterM3: verdict=weak bias=over n=2253 NRMSE=25732.52919771499
zone value vs totalOptimValue: verdict=insufficient bias=neutral n=2371 NRMSE=nan
gap conclusions (top 10)
siteId=160 value_col=waterM3 gap_days=632 2024-07-10 -> 2026-04-02
siteId=130 value_col=waterM3 gap_days=484 2024-06-30 -> 2025-10-26
siteId=180 value_col=waterM3 gap_days=449 2025-01-09 -> 2026-04-02
siteId=150 value_col=waterM3 gap_days=217 2024-06-16 -> 2025-01-18
siteId=190 value_